# Bharat-AI Pocket Engine — Google Colab & Kaggle Studio

Run the complete **Bharat-AI Pocket Engine** on free cloud GPUs (Colab T4 or Kaggle T4/P100):
- **Mode A (60 FPS GPU Batch Render)**: Renders walking, side-profile, gesture-rich talking-avatar MP4 films (`3:4`, `9:16`, or `16:9`) with Devanagari ASS captions, `-17 LUFS` audio, and NVIDIA NVENC.
- **Mode B (Live Web Studio)**: Host the full React + Express + FFmpeg web app on port `3000` inside Colab Preview or a Kaggle public URL.

**Quick Start (Colab):**
1. Make sure **GPU** is selected (`Runtime -> Change runtime type -> T4 GPU`).
2. Click **Runtime -> Run all** (if you saved from the app to Google Drive, this notebook will find your `.zip` automatically in `MyDrive/Bharat-AI-Colab/`).

In [ ]:
# @title 1. Check Cloud NVIDIA GPU & Install System Dependencies
!nvidia-smi
!apt-get update -qq && apt-get install -y -qq curl ffmpeg fonts-noto-core fonts-noto-extra libegl1 libgl1 libvulkan1 mesa-vulkan-drivers unzip > /dev/null
!curl -fsSL https://deb.nodesource.com/setup_22.x | bash - > /dev/null && apt-get install -y -qq nodejs > /dev/null
!node -v && ffmpeg -version | head -n 1
!pip install -q playwright edge-tts
!playwright install chromium

In [ ]:
# @title 2. Auto-Locate Project Bundle (Google Drive / Colab / Kaggle) & Install
import os, zipfile, glob
from pathlib import Path

def find_archives():
    return (
        glob.glob('/content/*.zip')
        + glob.glob('/content/*.tar.gz')
        + glob.glob('/kaggle/input/**/*.zip', recursive=True)
        + glob.glob('/kaggle/input/**/*.tar.gz', recursive=True)
        + glob.glob('/content/drive/MyDrive/Bharat-AI-Colab/*.zip')
        + glob.glob('/content/drive/MyDrive/Bharat-AI-Colab/*.tar.gz')
        + glob.glob('*.zip')
        + glob.glob('*.tar.gz')
    )

archives = find_archives()
if not archives and Path('/content').exists():
    try:
        from google.colab import drive
        print('Checking Google Drive (MyDrive/Bharat-AI-Colab)...')
        drive.mount('/content/drive', force_remount=False)
        archives = find_archives()
    except Exception as e:
        print('Drive mount skipped:', e)

if not archives:
    try:
        from google.colab import files
        print('Select bharat-ai-pocket-engine-colab-gpu.zip to upload...')
        uploaded = files.upload()
        archives = list(uploaded.keys())
    except ImportError:
        raise FileNotFoundError('Upload bharat-ai-pocket-engine-colab-gpu.zip into /content, Google Drive Bharat-AI-Colab, or Kaggle Input first!')

archive_path = Path(archives[0])
project_dir = Path('/content/bharat_ai_studio') if Path('/content').exists() else Path('/kaggle/working/bharat_ai_studio')
project_dir.mkdir(parents=True, exist_ok=True)

if archive_path.name.endswith('.tar.gz'):
    import tarfile
    with tarfile.open(archive_path, 'r:gz') as tf:
        tf.extractall(project_dir)
else:
    with zipfile.ZipFile(archive_path, 'r') as zf:
        zf.extractall(project_dir)

%cd {project_dir}
!npm install --silent
print('Project ready at:', project_dir, 'from', archive_path)

In [ ]:
# @title 3. (Optional) Launch Full Live Web Studio UI in Colab / Kaggle
# Leave RUN_WEB_APP = False if you only want to render your MP4 below.
RUN_WEB_APP = False  # @param {type:"boolean"}
GEMINI_API_KEY = ''  # @param {type:"string"}

if RUN_WEB_APP:
    import os, subprocess, time
    !npm run build
    if GEMINI_API_KEY:
        os.environ['GEMINI_API_KEY'] = GEMINI_API_KEY
    os.environ['NODE_ENV'] = 'production'
    os.environ['PORT'] = '3000'
    os.environ['DISABLE_HMR'] = 'true'
    web_proc = subprocess.Popen(['npx', 'tsx', 'server.ts'])
    time.sleep(3)
    try:
        from google.colab import output
        output.serve_kernel_port_as_window(3000)
    except ImportError:
        !curl -fsSL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
        !cloudflared tunnel --url http://127.0.0.1:3000

In [ ]:
# @title 4. Configure Gesture Boost & Inspect Exported Scenes
import json
from pathlib import Path

FPS = 60  # @param {type:"integer"}
GESTURE_BOOST = 1.45  # @param {type:"slider", min:1.0, max:1.8, step:0.05}
OUTPUT_MP4 = 'bharat_ai_gpu_60fps.mp4'

cfg_path = Path('colab_kaggle_gpu/scenes_config.json')
if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text(encoding='utf-8'))
    print(f"Loaded {len(cfg.get('scenes', []))} scenes · Format: {cfg.get('format')} · Language: {cfg.get('language')}")
else:
    raise FileNotFoundError('colab_kaggle_gpu/scenes_config.json not found in bundle.')

In [ ]:
# @title 5. Render 60 FPS MP4 on Cloud GPU (NVENC + ASS Captions + -17 LUFS Audio)
!python colab_kaggle_gpu/gpu_render.py --config colab_kaggle_gpu/scenes_config.json --output {OUTPUT_MP4} --fps {FPS} --gesture-boost {GESTURE_BOOST}

In [ ]:
# @title 6. Preview & Download Final MP4
from IPython.display import Video, display
display(Video(OUTPUT_MP4, embed=True, width=420))

try:
    from google.colab import files
    files.download(OUTPUT_MP4)
except ImportError:
    print('Kaggle Output saved at:', OUTPUT_MP4)